# Step 4 - EDA 2D

Looking at relationships between columns. Focused on what might actually matter:
- price vs surface
- price vs type of property
- price vs location (departement)
- price trend over time by type
- correlation between numerical columns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet('../data/cleaned.parquet')

# base filter: regular sales, no outliers, has a building
ventes = df[
    (df['nature_mutation'] == 'Vente') &
    (~df['is_outlier_valeur']) &
    (df['valeur_fonciere'] > 1000)  # remove symbolic transfers
].copy()

print(f'Working subset: {len(ventes):,} rows')

## Valeur par type de bien

In [ ]:
# boxplot by type_local - want to see if prices differ a lot between categories
types_keep = ['Maison', 'Appartement', 'Terrain', 'Local industriel. commercial ou assimilé']

fig, ax = plt.subplots(figsize=(10, 5))

data_by_type = [
    np.log10(ventes[ventes['type_local'] == t]['valeur_fonciere'] + 1)
    for t in types_keep
]

bp = ax.boxplot(data_by_type, labels=['Maison', 'Appartement', 'Terrain', 'Local comm.'],
                patch_artist=True, showfliers=False)

colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Distribution log10(valeur) par type de bien')
ax.set_ylabel('log10(€)')
plt.tight_layout()
plt.savefig('../outputs/plots/08_valeur_par_type.png', dpi=100)
plt.show()

# medians by type
for t in types_keep:
    med = ventes[ventes['type_local'] == t]['valeur_fonciere'].median()
    print(f'{t}: median = {med:,.0f}€')

## Prix au m² - Maison vs Appartement

In [ ]:
# price per m² only makes sense for buildings with known surface
bati = ventes[(ventes['surface_bati'] > 10) & (ventes['surface_bati'] <= 500)].copy()
bati['prix_m2'] = bati['valeur_fonciere'] / bati['surface_bati']

# remove obvious errors in prix_m2
bati = bati[(bati['prix_m2'] > 100) & (bati['prix_m2'] < 30000)]

print(bati.groupby('type_local')['prix_m2'].describe().round(0))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for t, color in [('Maison', 'steelblue'), ('Appartement', 'coral')]:
    subset = bati[bati['type_local'] == t]['prix_m2']
    ax.hist(subset, bins=100, alpha=0.6, label=t, color=color, edgecolor='none', density=True)

ax.set_title('Prix au m² - Maison vs Appartement')
ax.set_xlabel('€/m²')
ax.set_ylabel('Densité')
ax.legend()
ax.set_xlim(0, 10000)
plt.tight_layout()
plt.savefig('../outputs/plots/09_prix_m2_type.png', dpi=100)
plt.show()

# apartments skew higher per m², which makes sense
# but overlap is large - location probably matters more than type

## Valeur vs surface bâtie (scatter)

In [ ]:
# scatter: surface vs price - expecting positive correlation
# sample to avoid overplotting
sample = bati[bati['type_local'].isin(['Maison', 'Appartement'])].sample(15000, random_state=42)

fig, ax = plt.subplots(figsize=(9, 5))

for t, color in [('Maison', 'steelblue'), ('Appartement', 'coral')]:
    sub = sample[sample['type_local'] == t]
    ax.scatter(sub['surface_bati'], sub['valeur_fonciere'], 
               alpha=0.15, s=5, color=color, label=t)

ax.set_xlabel('Surface bâtie (m²)')
ax.set_ylabel('Valeur foncière (€)')
ax.set_title('Surface vs Valeur')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
ax.legend()
ax.set_xlim(0, 300)
ax.set_ylim(0, 1_200_000)
plt.tight_layout()
plt.savefig('../outputs/plots/10_surface_vs_valeur.png', dpi=100)
plt.show()

# correlation?
for t in ['Maison', 'Appartement']:
    sub = bati[bati['type_local'] == t]
    corr = sub['surface_bati'].corr(sub['valeur_fonciere'])
    print(f'{t}: corr surface/valeur = {corr:.3f}')

## Évolution du prix médian par année et type

In [ ]:
# did prices evolve differently for houses vs apartments?
pivot = (
    ventes[ventes['type_local'].isin(['Maison', 'Appartement', 'Terrain'])]
    .groupby(['annee', 'type_local'])['valeur_fonciere']
    .median()
    .unstack()
)
print(pivot)

fig, ax = plt.subplots(figsize=(9, 4))
pivot.plot(ax=ax, marker='o')
ax.set_title('Valeur médiane par année et type de bien')
ax.set_ylabel('€')
ax.set_xlabel('Année')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k€'))
plt.tight_layout()
plt.savefig('../outputs/plots/11_prix_evolution.png', dpi=100)
plt.show()

## Prix médian par département (top 20)

In [ ]:
# which departments are most expensive?
# filtering to maisons only to keep it comparable
maisons = ventes[ventes['type_local'] == 'Maison']

dept_stats = (
    maisons.groupby('code_departement')['valeur_fonciere']
    .agg(['median', 'count'])
    .query('count > 500')  # ignore tiny departments
    .sort_values('median', ascending=False)
    .head(20)
)
print(dept_stats)

fig, ax = plt.subplots(figsize=(12, 4))
dept_stats['median'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
ax.set_title('Valeur médiane des maisons par département (top 20)')
ax.set_xlabel('Département')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k€'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/plots/12_prix_dept.png', dpi=100)
plt.show()

# expecting 75, 92, 94 at the top (Paris + inner suburbs)

## Corrélations entre variables numériques

In [ ]:
num_cols = ['valeur_fonciere', 'surface_bati', 'nb_pieces', 'surface_terrain', 'nombre_lots']

corr = ventes[num_cols].corr()
print(corr.round(3))

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im)

ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=30, ha='right', fontsize=9)
ax.set_yticklabels(num_cols, fontsize=9)

for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)

ax.set_title('Matrice de corrélation')
plt.tight_layout()
plt.savefig('../outputs/plots/13_correlation.png', dpi=100)
plt.show()

# surface_bati and nb_pieces are correlated (makes sense - bigger = more rooms)
# valeur vs surface_terrain correlation probably low - terrain value depends on location more

## Nombre de pièces vs valeur (maisons)

In [ ]:
maison_pieces = ventes[
    (ventes['type_local'] == 'Maison') &
    (ventes['nb_pieces'].between(1, 10))
]

med_by_pieces = maison_pieces.groupby('nb_pieces')['valeur_fonciere'].median()
count_by_pieces = maison_pieces.groupby('nb_pieces').size()

print('Median valeur by nb_pieces (maisons):')
print(pd.DataFrame({'median_€': med_by_pieces, 'count': count_by_pieces}))

fig, ax = plt.subplots(figsize=(8, 4))
med_by_pieces.plot(kind='bar', ax=ax, color='coral', edgecolor='none')
ax.set_title('Valeur médiane des maisons par nombre de pièces')
ax.set_xlabel('Nb pièces')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k€'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../outputs/plots/14_valeur_pieces.png', dpi=100)
plt.show()

# expected a clear upward trend - more rooms = more expensive
# curious if it actually holds or flattens at some point